<a href="https://colab.research.google.com/github/jeffersonramelo/M-TODOS-QUANTITATIVOS/blob/main/testes_param%C3%A9tricos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Para a ANOVA de medidas repetidas, os dados precisam estar em um formato 'longo' (long format). Isso significa que cada medida para um sujeito deve estar em uma linha separada, com uma coluna indicando o tempo ou a condição da medida.

Teste T para 1 amostra, assumindo que a nota média da população é de 150

In [16]:
import pandas as pd

df1 = pd.read_csv('/content/DadosEmpresasDaBolsa_test 30 obs.csv')

df1

,Notas
0,160
1,160
2,157
3,163
4,163
5,155
6,165
7,168
8,163
9,170


In [17]:
df1['Notas'].describe()

,Notas
count,29.000000
mean,168.379310
std,11.377967
min,145.000000
25%,160.000000
50%,165.000000
75%,173.000000
max,193.000000


In [35]:
from scipy import stats

population_mean = 150
t_statistic, p_value = stats.ttest_1samp(df1['Notas'], population_mean)

print(f"T-statistic: {t_statistic:.2f}")
print(f"P-value: {p_value:.3f}")

alpha = 0.05 # nível de significância
if p_value < alpha:
    print("Rejeitamos a hipótese nula. Há evidências suficientes para dizer que a média amostral é significativamente diferente da média populacional de 150.")
else:
    print("Não rejeitamos a hipótese nula. Não há evidências suficientes para dizer que a média amostral é significativamente diferente da média populacional de 150.")

T-statistic: 8.70
P-value: 0.000

Interpretação (com base na hipótese nula):
Rejeitamos a hipótese nula. Há evidências suficientes para dizer que a média amostral é significativamente diferente da média populacional de 150.


Teste T para duas amostras independentes


In [24]:
import pandas as pd
import numpy as np
from scipy import stats

# Reload the correct dataframe for independent samples t-test
df2 = pd.read_csv('/content/DadosEmpresasDaBolsa_amostras independentes_test_t.csv', sep=';')

# Separate the data into two groups based on 'Posicao'
frente_notas = df2[df2['Posicao'] == 'Frente']['Notas']
fundo_notas = df2[df2['Posicao'] == 'Fundo']['Notas']

# Perform independent two-sample t-test
t_statistic_ind, p_value_ind = stats.ttest_ind(frente_notas, fundo_notas, equal_var=False) # Assuming unequal variances, which is often a safer default

print(f"T-statistic (Two-sample): {t_statistic_ind:.2f}")
print(f"P-value (Two-sample): {p_value_ind:.3f}")

alpha = 0.05 # Significance level

if p_value_ind < alpha:
    print("Rejeitamos a hipótese nula. Há evidências suficientes para dizer que as médias dos dois grupos são significativamente diferentes.")
else:
    print("Não rejeitamos a hipótese nula. Não há evidências suficientes para dizer que as médias dos dois grupos são significativamente diferentes.")

T-statistic (Two-sample): -1.87
P-value (Two-sample): 0.072
Não rejeitamos a hipótese nula. Não há evidências suficientes para dizer que as médias dos dois grupos são significativamente diferentes.


Anova de uma Via, quando existe mais de dois grupos para comparar. É usada quado o que o teste t para duas amostras não é capaz de fazer.

In [27]:
import pandas as pd
from scipy import stats

# Load the dataset with 'ISO-8859-1' encoding and semicolon separator
df_anova = pd.read_csv('/content/Dados anova de uma via.csv', encoding='ISO-8859-1', sep=';')

# Group the 'LiqCor' values by 'Tamanho'
groups = [df_anova['LiqCor'][df_anova['Tamanho'] == t] for t in df_anova['Tamanho'].unique()]

# Perform one-way ANOVA
f_statistic, p_value_anova = stats.f_oneway(*groups)

print(f"F-statistic (ANOVA): {f_statistic:.2f}")
print(f"P-value (ANOVA): {p_value_anova:.3f}")

alpha = 0.05 # Significance level

if p_value_anova < alpha:
    print("Rejeitamos a hipótese nula. Há evidências suficientes para dizer que há uma diferença significativa entre as médias de 'LiqCor' para os diferentes grupos de 'Tamanho'.")
else:
    print("Não rejeitamos a hipótese nula. Não há evidências suficientes para dizer que há uma diferença significativa entre as médias de 'LiqCor' para os diferentes grupos de 'Tamanho'.")

F-statistic (ANOVA): 0.44
P-value (ANOVA): 0.817
Não rejeitamos a hipótese nula. Não há evidências suficientes para dizer que há uma diferença significativa entre as médias de 'LiqCor' para os diferentes grupos de 'Tamanho'.


Anova de duas vias, quando existe mais de dois grupos para comparar e duas variáveis (vias) para analizar. É usada quando a Anova de uma via não é suficiente.

In [57]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Carregar os dados
df = pd.read_csv(
    '/content/Dados anova de duas vias.csv',
    sep=';',
    encoding='ISO-8859-1'
)

# ANOVA de duas vias
modelo = ols(
    'retorno ~ C(porte) * C(sexoCEO)',
    data=df
).fit()

anova = sm.stats.anova_lm(modelo, typ=2)

print("\nRESULTADOS DA ANOVA DE DUAS VIAS\n")
print(anova)

alpha = 0.05

print("\nINTERPRETAÇÃO DOS RESULTADOS\n")

# Efeito do porte
p_porte = anova.loc['C(porte)', 'PR(>F)']

if p_porte < alpha:
    print(f"PORTE: Rejeitamos H0 (p = {p_porte:.4f}).")
    print("Há evidências suficientes para afirmar que pelo menos um dos grupos de porte possui média de retorno diferente dos demais.\n")
else:
    print(f"PORTE: Não rejeitamos H0 (p = {p_porte:.4f}).")
    print("Não há evidências suficientes para afirmar que as médias de retorno diferem entre os grupos de porte.\n")

# Efeito do sexo do CEO
p_sexo = anova.loc['C(sexoCEO)', 'PR(>F)']

if p_sexo < alpha:
    print(f"SEXO DO CEO: Rejeitamos H0 (p = {p_sexo:.4f}).")
    print("Há evidências suficientes para afirmar que o sexo do CEO influencia o retorno médio.\n")
else:
    print(f"SEXO DO CEO: Não rejeitamos H0 (p = {p_sexo:.4f}).")
    print("Não há evidências suficientes para afirmar que o sexo do CEO influencia o retorno médio.\n")

# Interação
p_interacao = anova.loc['C(porte):C(sexoCEO)', 'PR(>F)']

if p_interacao < alpha:
    print(f"INTERAÇÃO PORTE × SEXO DO CEO: Rejeitamos H0 (p = {p_interacao:.4f}).")
    print("Há evidências suficientes para afirmar que o efeito do porte sobre o retorno depende do sexo do CEO.\n")
else:
    print(f"INTERAÇÃO PORTE × SEXO DO CEO: Não rejeitamos H0 (p = {p_interacao:.4f}).")
    print("Não há evidências suficientes para afirmar que existe interação entre porte e sexo do CEO.\n")


RESULTADOS DA ANOVA DE DUAS VIAS

                         sum_sq    df         F    PR(>F)
C(porte)               9.523786   2.0  1.155245  0.330626
C(sexoCEO)             1.582453   1.0  0.383906  0.540912
C(porte):C(sexoCEO)    4.742015   2.0  0.575211  0.569584
Residual             107.171365  26.0       NaN       NaN

INTERPRETAÇÃO DOS RESULTADOS

PORTE: Não rejeitamos H0 (p = 0.3306).
Não há evidências suficientes para afirmar que as médias de retorno diferem entre os grupos de porte.

SEXO DO CEO: Não rejeitamos H0 (p = 0.5409).
Não há evidências suficientes para afirmar que o sexo do CEO influencia o retorno médio.

INTERAÇÃO PORTE × SEXO DO CEO: Não rejeitamos H0 (p = 0.5696).
Não há evidências suficientes para afirmar que existe interação entre porte e sexo do CEO.



Teste T pareado

In [10]:
import pandas as pd

df = pd.read_csv('/content/DadosEmpresasDaBolsa_test t pareado.csv', sep=';')

df

,Nota_antes,Nota_depois
0,3,3
1,4,5
2,3,1
3,7,7
4,1,3
...,...,...
270,1,2
271,1,1
272,3,1
273,1,2


In [12]:
from scipy import stats

# Perform paired t-test
t_statistic_paired, p_value_paired = stats.ttest_rel(df['Nota_antes'], df['Nota_depois'])

print(f"T-statistic (Paired): {t_statistic_paired:.2f}")
print(f"P-value (Paired): {p_value_paired:.3f}")

alpha = 0.05 # Significance level

if p_value_paired < alpha:
    print("Rejeitamos a hipótese nula. Há evidências suficientes para dizer que há uma diferença significativa entre as médias das notas antes e depois.")
else:
    print("Não rejeitamos a hipótese nula. Não há evidências suficientes para dizer que há uma diferença significativa entre as médias das notas antes e depois.")

T-statistic (Paired): 3.68
P-value (Paired): 0.000
Rejeitamos a hipótese nula. Há evidências suficientes para dizer que há uma diferença significativa entre as médias das notas antes e depois.


Anova de uma via com medidas repetidas usado quanto o tempo utrapassa o antes e depois do teste t pareado

In [60]:
import pandas as pd
from statsmodels.stats.anova import AnovaRM

# Carregar os dados em formato longo
df = pd.read_csv('/content/anova_medidas_repetidas_formato_longo.csv')

# Garantir que a variável dependente é numérica
df['valor'] = df['valor'].astype(float)

# ANOVA de uma via com medidas repetidas
resultado = AnovaRM(
    data=df,
    depvar='valor',
    subject='empresa',
    within=['trimestre']
).fit()

# Mostrar resultado
print("\nRESULTADO DA ANOVA DE MEDIDAS REPETIDAS\n")
print(resultado)

# Interpretação
p_valor = resultado.anova_table.loc['trimestre', 'Pr > F']
alpha = 0.05

print("\nINTERPRETAÇÃO\n")

print("H0: As médias são iguais ao longo dos trimestres.")
print("H1: Pelo menos uma média é diferente ao longo dos trimestres.\n")

if p_valor < alpha:
    print(f"Rejeitamos H0, pois p = {p_valor:.4f} < 0.05.")
    print("Há evidências de diferença significativa entre os trimestres.")
else:
    print(f"Não rejeitamos H0, pois p = {p_valor:.4f} >= 0.05.")
    print("Não há evidências de diferença significativa entre os trimestres.")


RESULTADO DA ANOVA DE MEDIDAS REPETIDAS

                 Anova
          F Value Num DF  Den DF  Pr > F
----------------------------------------
trimestre 39.7647 3.0000 147.0000 0.0000


INTERPRETAÇÃO

H0: As médias são iguais ao longo dos trimestres.
H1: Pelo menos uma média é diferente ao longo dos trimestres.

Rejeitamos H0, pois p = 0.0000 < 0.05.
Há evidências de diferença significativa entre os trimestres.
